In [0]:
%sql
-- 4.3 MISSED CALLS METRIC — MISSED_CALLS_14_DAYS (CRM signal)
-- Spec: count UNIQUE ACTIVITY_IDs in the previous 14 days where
-- TYPE = 'Call', DIRECTION = 'inbound', DISPOSITION = 'no-answer';
-- RETURN 0 when none qualify.
-- Population: every lead in gold.lead_dim (all-leads spine, 3_04 v4
-- convention — coaching filter applied downstream at 3_13/3_14 profile
-- assembly). Leads with no qualifying call get an explicit 0 row, never a
-- missing row / NULL (previously only leads WITH calls appeared — 244 rows —
-- so every join returned NULL for the rest, violating the spec's 0 rule).
-- FIX v2 (Nov 12 2026): switched COUNT(*) over an id_rn dedup window to
-- COUNT(DISTINCT id) — the spec-faithful unique count and a no-op under the
-- one-row-per-id silver contract; window hardened to BETWEEN 0 AND 14
-- (>= 0 guards against future-stamped activity_at rows).
-- Vocabulary verified Nov 12 2026 on silver: Call direction ∈ {outbound,
-- inbound} (exact lowercase — unlike Email's outgoing/incoming); Call
-- disposition ∈ {answered, no-answer, busy, error, vm-answer, vm-left,
-- blocked, NULL} — 'no-answer' spelled exactly per spec. Inbound+no-answer =
-- 3,932 activities all-history; 244 leads / 287 calls in the current window.
-- NOTE (deferred job cleanup): this table now sources gold.lead_dim — the
-- gold_missed_calls job task must gain depends_on gold_lead_dim.
CREATE OR REPLACE TABLE crm_ingestion.gold.missed_calls_14_days AS (
WITH missed_calls AS (
    SELECT
        lead_id,
        COUNT(DISTINCT id) AS missed_calls_14_days
    FROM
        crm_ingestion.silver.lead_activites_processed
    WHERE
        type = 'Call'
        AND direction = 'inbound'
        AND disposition = 'no-answer'
        AND DATEDIFF(CURRENT_DATE, DATE(activity_at)) BETWEEN 0 AND 14
    GROUP BY
        lead_id
)
SELECT
    l.lead_id,
    COALESCE(m.missed_calls_14_days, 0) AS missed_calls_14_days
FROM
    crm_ingestion.gold.lead_dim l
    LEFT JOIN missed_calls m ON l.lead_id = m.lead_id
)